# 📘 Guía explicativa del proyecto — *Consumo de Materiales / Minería*

**Para:** perfil de gestión (Project Manager) con poco conocimiento técnico.
**Qué es esto:** un cuaderno (*notebook* estilo Jupyter) que te explica, **paso a paso y con ejemplos que puedes ejecutar**, cómo funciona la aplicación web y, sobre todo, las tres cosas que te interesan para la defensa:

1. El **módulo de seguridad de acceso (RBAC)**: usuarios, roles, contraseñas y sesiones.
2. Las **consultas parametrizadas** que protegen contra la **inyección SQL**.
3. La **normalización** de la base de datos hasta la **3ª Forma Normal (3FN)**.

Y además responde a tu duda concreta: **por qué “no te aparece” el bloqueo de 24 horas tras 5 intentos fallidos** (spoiler: sí está programado y funciona; lo que ves es otra capa que lo tapa — lo demostramos con código más abajo).

> 💡 **Cómo se lee este cuaderno:** las celdas de texto (como esta) explican; las celdas grises con código se pueden **ejecutar** para ver el resultado en vivo. No necesitas entender el código: cada bloque va acompañado de una explicación en lenguaje sencillo.

## 🗂️ Índice

1. ¿Qué es este proyecto y cómo está armado?
2. Cómo abrir y usar este cuaderno en Jupyter
3. Poner en marcha el proyecto paso a paso (con celdas ejecutables)
4. **Errores encontrados y cómo se corrigieron** (bloqueo de 5 intentos / 24 h, historial vacío, entorno)
5. Objetivo específico 1 — Módulo de autenticación y autorización (RBAC)
6. Objetivo específico 2 — Consultas parametrizadas vs Inyección SQL
7. Objetivo específico 3 — Normalización hasta 3FN
8. Mapa: tus conclusiones y recomendaciones ↔ el código
9. Glosario para perfil no técnico


## 1. ¿Qué es este proyecto y cómo está armado?

Es una **aplicación web** para controlar el **consumo de materiales** de una planta (diseños de mezcla, despachos, inventario, historial). Por dentro tiene dos grandes piezas:

- **El servidor (backend):** escrito en **Python** con el framework **Flask**. Es quien recibe las peticiones, valida usuarios, aplica los permisos y habla con la base de datos.
- **La base de datos:** un archivo **SQLite** (`db/gestion_materiales.db`). Es una base de datos relacional guardada en un solo archivo, ideal para proyectos de este tamaño.

Piensa en el proyecto como una **oficina**:

| Pieza del código | Analogía de oficina | Para qué sirve |
|---|---|---|
| `app.py` | La **recepción** | Recibe cada visita (petición web) y la manda a la ventanilla correcta |
| `auth/` | El **guardia de seguridad** | Revisa identidad (login) y permisos (rol) |
| `services/` | Los **departamentos** | Inventario, despachos, historial, dashboard: la lógica del negocio |
| `db/` y `utils/db.py` | El **archivo/bodega** | Crear la base, guardar y leer datos |
| `templates/` y `static/` | Las **salas de atención** | Las pantallas (HTML) y botones (JavaScript) que ve el usuario |
| `docs/` | Los **planos** | Documentación y diagramas de la base de datos |

**Los 3 roles del sistema** (esto es el corazón del RBAC, lo vemos en detalle luego):

- **Admin** → acceso total (ver, crear, editar, borrar, gestionar usuarios).
- **Operador** → puede ver y registrar (inventario e historial), pero no administra usuarios.
- **Visualizador** → solo puede ver el panel principal (*dashboard*).

## 2. Cómo abrir y usar este cuaderno en Jupyter

Un **notebook Jupyter** es un documento que mezcla texto explicativo con celdas de código que puedes ejecutar. Este archivo (`GUIA_EXPLICATIVA_PROYECTO.ipynb`) está **dentro de la carpeta del programa**, así que puede “tocar” el código real y demostrarte cosas en vivo.

**Opción A — Con VS Code (lo más simple si ya lo usas):**
1. Abre la carpeta `consumo-materiales-mineria` en VS Code.
2. Haz doble clic en `GUIA_EXPLICATIVA_PROYECTO.ipynb`.
3. Si te pide elegir un *kernel* (el “motor” de Python), elige tu Python 3.
4. Ejecuta cada celda con el botón ▶ de la izquierda (o `Shift + Enter`).

**Opción B — Con Jupyter en el navegador:**
1. Abre una terminal (símbolo del sistema) **dentro** de la carpeta `consumo-materiales-mineria`.
2. Instala Jupyter una sola vez: `pip install notebook`
3. Ejecútalo: `jupyter notebook`
4. Se abre el navegador; haz clic en `GUIA_EXPLICATIVA_PROYECTO.ipynb`.
5. Ejecuta celdas con `Shift + Enter`.

> ✅ **Regla de oro:** ejecuta las celdas **en orden, de arriba hacia abajo**. Algunas preparan cosas que las siguientes necesitan.

## 3. Poner en marcha el proyecto paso a paso

Aquí dejamos celdas **ejecutables** que preparan todo. Ejecútalas en orden.

### Paso 3.1 — Instalar las dependencias
El proyecto necesita algunas librerías de Python (Flask, etc.). Esta celda las instala automáticamente leyendo el archivo `requirements.txt`.

In [ ]:
# Instala las librerías que el proyecto necesita (una sola vez).
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("✅ Dependencias instaladas.")

### Paso 3.2 — Crear el archivo de configuración `.env`
El proyecto **exige** una clave secreta (`SECRET_KEY`) para firmar las sesiones de forma segura. Si el archivo `.env` no existe, el servidor **no arranca** a propósito (es una buena práctica de seguridad). Esta celda crea el `.env` con una clave aleatoria fuerte.

In [ ]:
import os, secrets
if not os.path.exists(".env"):
    with open(".env", "w", encoding="utf-8") as f:
        f.write("SECRET_KEY=" + secrets.token_hex(32) + "\n")  # clave aleatoria de 64 caracteres
        f.write("DB_PATH=db/gestion_materiales.db\n")
        f.write("FLASK_DEBUG=True\n")
    print("✅ Archivo .env creado con una SECRET_KEY nueva y aleatoria.")
else:
    print("ℹ️  Ya existe un .env, no lo modifico.")

### Paso 3.3 — Construir la base de datos
La base de datos **no viene incluida** en el repositorio (por seguridad, está en `.gitignore`). Se **construye** ejecutando unos scripts. Esta celda:

- crea el **esquema** (todas las tablas, incluida `intentos_login` del bloqueo), 
- **puebla los insumos** (materiales),
- crea los **usuarios de arranque**, y
- se asegura de la tabla de **intentos de login**.

> 🔎 Nota: hay un cuarto script, `db/03_cargar_datos_iniciales.py`, que carga datos históricos reales desde un Excel. Ese necesita la librería `pandas` (no incluida por defecto). Es **opcional** para esta guía; si lo quieres, primero ejecuta `pip install pandas`.

In [ ]:
import sys, subprocess
scripts = [
    "db/01_crear_esquema.py",       # crea TODAS las tablas (incl. intentos_login)
    "db/02_poblar_insumos.py",      # carga el catálogo de materiales
    "scripts/crear_usuarios.py",    # crea admin / operador / visor
    "scripts/crear_tabla_intentos.py",  # asegura la tabla del bloqueo (idempotente)
]
for s in scripts:
    print("\n>>> Ejecutando:", s)
    subprocess.run([sys.executable, s])

### Paso 3.4 — Arrancar el servidor y abrir la app
El servidor web se arranca con `python app.py`. **Ojo:** ese comando se queda “corriendo” (ocupa la terminal), por eso **no** conviene ejecutarlo dentro de una celda del cuaderno (bloquearía el notebook). Hazlo así:

1. Abre una terminal dentro de la carpeta `consumo-materiales-mineria`.
2. Ejecuta: `python app.py`
3. Verás algo como `Running on http://127.0.0.1:5000`.
4. Abre esa dirección en el navegador.

**Usuarios de prueba** (creados en el paso 3.3):

| Usuario | Contraseña | Rol |
|---|---|---|
| `admin` | `Admin123!` | Admin |
| `operador` | `Operador123!` | Operador |
| `visor` | `Visor123!` | Visualizador |

> ⚠️ Son credenciales **de arranque/demostración**. En un entorno real deben cambiarse.

## 4. 🐛 Errores encontrados y cómo se corrigieron

> *"Después de 5 intentos no me sale el mensaje de bloqueo de 24 horas. Además, al recargar la
> página el contador se reinicia y puedo entrar de nuevo. Y el historial no se ve cuando está
> subido a la web."*

Esta sección documenta **cinco errores reales** que impedían que el sistema funcionara como estaba
diseñado, con el diagnóstico de cada uno y el arreglo aplicado. Es la sección más útil para la
defensa, porque muestra el ciclo completo: *síntoma → causa raíz → corrección → verificación*.

| # | Síntoma que veía el usuario | Causa raíz | Dónde se arregló |
|---|---|---|---|
| 1 | No aparece el mensaje de bloqueo tras 5 intentos | El límite "5 por minuto" se disparaba **antes** y devolvía un error 429 en HTML | `app.py` |
| 2 | El aviso de bloqueo llegaba tarde (al 6º intento) | El 5º intento guardaba el bloqueo pero respondía "contraseña incorrecta" | `auth/login.py` |
| 3 | **Al recargar (F5) el contador parecía reiniciarse** | El bloqueo solo vivía en el navegador (JavaScript), no se consultaba al servidor | `app.py`, `templates/login.html`, `auth/login.py` |
| 4 | El historial sale vacío en la web | La base de datos está en `.gitignore`: **nunca se sube** al servidor | Procedimiento de despliegue |
| 5 | `pip install -r requirements.txt` falla entero | `numpy>=1.24,<2.0` no compila en Python 3.13 | `requirements.txt` |

Un dato importante que conviene decir en la defensa: **el motor del bloqueo siempre estuvo bien
programado**. El contador y la fecha de desbloqueo se guardan en la base de datos (tabla
`intentos_login`), no en memoria, así que sobreviven a reinicios del servidor. Los errores 1, 2 y 3
no estaban en *la lógica de seguridad*, sino en **las capas que la rodean**: el límite por minuto que
la tapaba, el momento en que se avisaba, y la interfaz que se olvidaba del estado al recargar.


---

### Error 1 — El límite "5 por minuto" tapaba el mensaje de bloqueo

**Síntoma.** Fallabas 5 veces la contraseña y en vez del mensaje de las 24 horas salía
*"Error de conexión con el servidor"*, o nada útil.

**Causa raíz.** En `app.py` la ruta del login tenía **dos candados que competían entre sí**:

```python
@app.route("/api/login", methods=["POST"])
@limiter.limit("5 per minute")   # ← candado A: máximo 5 peticiones por minuto
def api_login():
    ...
```

El candado A (Flask-Limiter, protección por *ritmo*) saltaba en la **misma petición** en la que el
candado B (el bloqueo de 24 h, protección por *cantidad*) debía avisar. Como el límite por minuto
tiene prioridad, el servidor respondía **HTTP 429 con una página HTML**, y la pantalla de login
—que espera JSON— reventaba al intentar leerla y mostraba *"Error de conexión"*.

Resultado: el bloqueo de 24 h **sí se activaba** en la base de datos, pero el usuario **nunca lo veía**.

**Corrección aplicada** (en `app.py`):

```python
@app.route("/api/login", methods=["POST"])
@csrf.exempt
# el limite por minuto es solo un freno anti-flood; la defensa real contra
# fuerza bruta es el bloqueo de 24h en auth/login.py. lo dejamos por encima de
# MAX_INTENTOS (5) para que el usuario alcance a ver el mensaje de bloqueo en
# vez de chocar antes con un 429 del rate limiter.
@limiter.limit("20 per minute")
def api_login():
    ...
```

Y para que un 429 nunca vuelva a romper la pantalla, ahora se devuelve **JSON** en lugar de HTML:

```python
@app.errorhandler(429)
def demasiadas_peticiones(e):
    """Devuelve JSON (no el HTML por defecto) cuando salta el rate limiter."""
    return jsonify({
        "ok": False,
        "error": "Demasiadas peticiones seguidas. Espera un momento e intenta de nuevo.",
    }), 429
```

> 💡 **Idea clave para la defensa:** no se eliminó el límite por minuto (sigue frenando ataques
> automatizados muy rápidos), sino que se **subió por encima de los 5 intentos** para que los dos
> candados dejen de pisarse. Cada uno protege de algo distinto: uno del *ritmo*, otro de la *cantidad*.


---

### Error 2 — El aviso de bloqueo llegaba un intento tarde

**Síntoma.** El 5º intento fallido decía *"Usuario o contraseña incorrectos"* y el mensaje de
bloqueo solo aparecía en el **6º**. Confuso: el sistema dice que bloquea a los 5, pero avisa al 6.

**Causa raíz.** En `auth/login.py`, la función `_registrar_fallo()` **guardaba** el bloqueo en la base
de datos, pero no le decía a quien la llamaba que acababa de bloquear. Así que `autenticar()` seguía
su curso y devolvía el mensaje genérico de credenciales incorrectas. El bloqueo solo se detectaba en
la petición *siguiente*, cuando `esta_bloqueado()` lo leía de la base de datos.

**Corrección aplicada.** `_registrar_fallo()` ahora devuelve **cuántos intentos lleva** y **si acaba de
bloquear**, y `autenticar()` avisa en el acto:

```python
if not credenciales_ok:
    intentos, bloqueado_ahora = _registrar_fallo(username, ip)

    # si este mismo intento fue el que alcanzo el limite, avisamos del
    # bloqueo aqui mismo en vez de esperar al siguiente intento.
    if bloqueado_ahora:
        return {
            "ok": False,
            "bloqueado": True,
            "error": mensaje_bloqueo(BLOQUEO_SEGUNDOS),
        }

    restantes = max(0, MAX_INTENTOS - intentos)
    return {
        "ok": False,
        "error": (
            "Usuario o contraseña incorrectos. "
            f"Te quedan {restantes} intento(s) antes del bloqueo de 24 horas."
        ),
    }
```

De paso se arreglaron **dos problemas de mensajería**:

- El texto decía *"Intenta de nuevo en ~1440 min"*. Nadie reconoce 1440 minutos como "24 horas".
  Ahora `mensaje_bloqueo()` lo expresa **en horas**.
- Los intentos previos ahora **avisan cuántos quedan**, así el usuario ve venir el bloqueo en lugar
  de que le caiga por sorpresa.


---

### Error 3 — ⭐ Al recargar la página (F5), el bloqueo "desaparecía"

Este es el error **más interesante de los tres**, y el más importante de entender para la defensa,
porque enseña un principio central de la seguridad web.

**Síntoma.** Te bloqueabas tras 5 intentos, pulsabas **F5**, y la pantalla volvía a aparecer limpia:
botón habilitado, sin mensaje de error. Parecía que **el contador se había reiniciado** y que podías
seguir intentando.

**Causa raíz.** El bloqueo se estaba pintando **solo en el navegador**. La función `mostrarError()` de
`templates/login.html` deshabilitaba el botón *en memoria*:

```javascript
if (bloqueado) {
  btnIngresar.disabled = true;        // ← esto vive solo en esta carga de la página
  btnIngresar.textContent = "Bloqueado";
}
```

Al pulsar F5, el navegador **reconstruye la página desde cero**: el botón vuelve a nacer habilitado y
el mensaje de error desaparece. El estado del bloqueo vivía en un sitio que **el usuario puede borrar
cuando quiera**.

> ⚠️ **El matiz que hay que entender bien:** el contador **NUNCA se reseteó de verdad**. La base de
> datos seguía teniendo el bloqueo, y el servidor seguía rechazando el login. Lo que se reseteaba era
> **la apariencia**. Era un error de interfaz que *parecía* un agujero de seguridad.
>
> Lo comprobamos: estando bloqueado, si mandas la contraseña **correcta**, el servidor responde
> **HTTP 423 (bloqueado)** igual. Nunca hubo forma real de entrar.

**Corrección aplicada.** El principio: **el estado de seguridad no puede vivir en el navegador; tiene
que preguntarse al servidor en cada carga.** Se hizo en tres piezas.

**(a) `auth/login.py`** — nueva función que consulta si una IP tiene un bloqueo vigente, sin necesidad
de que el usuario haya escrito su nombre todavía (al recargar, el formulario está vacío):

```python
def estado_bloqueo_ip(ip: str) -> tuple[bool, int]:
    """
    dice si hay algun bloqueo activo desde esta ip, sin saber el usuario.

    la usa el login al cargar la pagina: si el navegador recarga (F5) el
    formulario se dibuja limpio y el usuario cree que el contador se reinicio.
    consultando esto al arrancar la pagina el bloqueo se pinta de una.
    """
    sufijo = f"%@{ip or 'desconocida'}"
    ahora = time.time()

    with conectar() as conexion:
        cursor = conexion.cursor()
        cursor.execute(
            """
            SELECT MAX(bloqueado_hasta) AS hasta
            FROM intentos_login
            WHERE clave LIKE ? AND bloqueado_hasta > ?
            """,
            (sufijo, ahora),   # ← parametrizada, igual que todo el resto
        )
        fila = cursor.fetchone()
    ...
```

**(b) `app.py`** — la ruta `/login` consulta ese estado y **manda el formulario ya deshabilitado desde
el servidor**:

```python
@app.route("/login")
def login():
    """
    Pinta el login. Si la IP ya está bloqueada por fuerza bruta, la plantilla
    se renderiza con el formulario deshabilitado desde el servidor.

    Esto es lo que evita que un F5 "reinicie" el bloqueo: el estado no vive en
    el navegador, se vuelve a leer de la BD en cada carga de la página.
    """
    bloqueado, restante = estado_bloqueo_ip(get_remote_address())
    return render_template(
        "login.html",
        bloqueado=bloqueado,
        mensaje_bloqueo=mensaje_bloqueo(restante) if bloqueado else "",
        segundos_restantes=restante,
    )
```

**(c) `templates/login.html`** — el botón se dibuja según lo que diga el servidor, no según lo que
recuerde el navegador:

```html
<button class="btn" type="submit" id="btnIngresar"
  {% if bloqueado %}disabled style="opacity:.55; cursor:not-allowed;"{% endif %}>
  {% if bloqueado %}Bloqueado{% else %}Ingresar{% endif %}
</button>
```

**Dos agujeros adicionales que hubo que tapar** (y que son buen material para la defensa, porque son
justo los que se escapan):

1. **La caché del navegador.** Si `/login` se guarda en caché, un F5 podría reusar la copia vieja —con
   el botón habilitado— **sin preguntarle nada al servidor**. Se añadió `/login` a las rutas que se
   sirven con `Cache-Control: no-store`:

   ```python
   # /login va sin caché: si no, un F5 podría reusar la copia cacheada (con el
   # formulario habilitado) y parecería que el bloqueo de 24h se reinició.
   rutas_protegidas = ("/login", "/dashboard", "/registro", "/inventario", "/historial", "/ml")
   ```

2. **El botón "atrás" del navegador.** Al volver atrás, el navegador puede restaurar la página desde
   su memoria (*bfcache*) **sin recargarla**, devolviendo el botón habilitado. Se añadió una
   revalidación contra el servidor:

   ```javascript
   window.addEventListener("pageshow", async (e) => {
     if (!e.persisted) return;               // solo si vino de la caché
     const r = await fetch("/api/estado_bloqueo", { cache: "no-store" });
     const d = await r.json();
     if (d.bloqueado) mostrarError(d.error, true);
   });
   ```

> 🎓 **La lección de seguridad (dilo así en la defensa):** *nunca confíes en el cliente*. El navegador
> es territorio del usuario: puede recargar, ir atrás, o abrir las herramientas de desarrollo y borrar
> el `disabled` del botón a mano. Por eso el frontend deshabilitado es solo **comodidad visual**, y la
> defensa real es que **el servidor rechaza el login con HTTP 423 pase lo que pase** — incluso con la
> contraseña correcta. Las dos capas hacen falta: una para que el usuario entienda, otra para que el
> atacante no pase.


---

### Verificación: la prueba que demuestra que los tres errores están corregidos

Esta celda hace la prueba completa **contra el servidor real** (arráncalo antes con `python app.py`):
falla 5 veces, **simula el F5** volviendo a pedir la página de login, y comprueba que sigue bloqueado
incluso mandando la contraseña correcta.


In [ ]:
# Prueba de extremo a extremo del bloqueo. Requiere el servidor corriendo:
#     python app.py      (en otra terminal)
import json, re, sqlite3, urllib.request, urllib.error

BASE = "http://127.0.0.1:5000"

# Partimos de cero: borramos cualquier bloqueo anterior.
con = sqlite3.connect("db/gestion_materiales.db")
con.execute("DELETE FROM intentos_login"); con.commit(); con.close()
print("== estado limpio ==\n")

def login(password):
    """Manda un intento de login y devuelve (codigo_http, mensaje)."""
    req = urllib.request.Request(
        f"{BASE}/api/login",
        data=json.dumps({"usuario": "admin", "password": password}).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        r = urllib.request.urlopen(req)
        return r.status, "LOGIN CORRECTO"
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read().decode())["error"]

def recargar_login():
    """Simula pulsar F5: vuelve a pedir la pagina /login al servidor."""
    r = urllib.request.urlopen(f"{BASE}/login")
    html = r.read().decode()
    boton = re.search(r'id="btnIngresar".*?</button>', html, re.S).group(0)
    return ("disabled" in boton), r.headers.get("Cache-Control")

# 1) Cinco intentos fallidos.
for i in range(1, 6):
    codigo, msg = login("CLAVE_INCORRECTA")
    print(f"Intento {i}: HTTP {codigo} | {msg}")

# 2) El F5 que antes "reiniciaba" el contador.
print("\n*** SIMULANDO F5 (nueva carga de /login) ***")
deshabilitado, cache = recargar_login()
print(f"   Boton deshabilitado por el servidor: {deshabilitado}")
print(f"   Cache-Control: {cache}")

# 3) La prueba de fuego: la contrasena CORRECTA, estando bloqueado.
print("\n*** Intento con la contrasena CORRECTA estando bloqueado ***")
codigo, msg = login("admin123")
print(f"   HTTP {codigo} | {msg}")
print("\n[OK] Ni recargando, ni con la clave correcta se puede entrar.")


Al ejecutarla, la salida es exactamente esta:

```
== estado limpio ==

Intento 1: HTTP 401 | Usuario o contraseña incorrectos. Te quedan 4 intento(s) antes del bloqueo de 24 horas.
Intento 2: HTTP 401 | Usuario o contraseña incorrectos. Te quedan 3 intento(s) antes del bloqueo de 24 horas.
Intento 3: HTTP 401 | Usuario o contraseña incorrectos. Te quedan 2 intento(s) antes del bloqueo de 24 horas.
Intento 4: HTTP 401 | Usuario o contraseña incorrectos. Te quedan 1 intento(s) antes del bloqueo de 24 horas.
Intento 5: HTTP 423 | Cuenta bloqueada por 24 horas tras 5 intentos fallidos. Inténtalo de nuevo en ~24 h.

*** SIMULANDO F5 (nueva carga de /login) ***
   Boton deshabilitado por el servidor: True
   Cache-Control: no-store, no-cache, must-revalidate, max-age=0

*** Intento con la contrasena CORRECTA estando bloqueado ***
   HTTP 423 | Cuenta bloqueada por 24 horas tras 5 intentos fallidos. Inténtalo de nuevo en ~24 h.

[OK] Ni recargando, ni con la clave correcta se puede entrar.
```

Léelo así, que es lo que hay que defender:

- Los intentos **1 a 4** avisan cuántos quedan (antes solo decían "incorrectos", sin más).
- El intento **5** ya devuelve **HTTP 423** con el mensaje de las **24 horas** (antes salía en el 6º, y
  encima tapado por el error 429 del límite por minuto).
- Tras el **F5**, el servidor manda el botón **deshabilitado** y sin caché (antes el formulario volvía
  a estar disponible).
- Con la contraseña **correcta**, sigue devolviendo **HTTP 423**. Esta última línea es la que prueba
  que la seguridad **no depende del navegador**.

También se comprobó el caso contrario: cuando **expiran las 24 horas**, el formulario se vuelve a
habilitar solo y el login correcto pasa con HTTP 200 — nadie queda encerrado para siempre.

> 🔎 **Nota sobre `/api/estado_bloqueo`:** este endpoint nuevo solo dice *si esta IP está bloqueada y
> cuánto le queda*. **No revela** si un usuario existe ni cuántos intentos lleva, así que no le da
> información útil a un atacante.


---

### Error 4 — El historial no se ve cuando el proyecto está subido a la web

**Síntoma.** En local el historial funciona, pero **subido al servidor sale vacío**.

**Causa raíz.** Esto **no es un error del código**: es un problema de *despliegue*. El archivo
[`.gitignore`](.gitignore) contiene esta línea:

```
# Base de datos
*.db
```

Eso significa que **la base de datos nunca se sube** al repositorio (y está bien que sea así: un `.db`
es un archivo binario pesado, cambia constantemente, y puede contener datos sensibles). Pero la
consecuencia es que **en el servidor web no hay base de datos**, o hay una vacía. Sin datos, el
historial no tiene nada que mostrar.

**Corrección.** Después de desplegar, hay que **construir la base de datos en el servidor** ejecutando
los mismos tres scripts del Paso 3.3:

```bash
python db/01_crear_esquema.py         # crea las tablas
python db/02_poblar_insumos.py        # catálogo de insumos + usuarios iniciales
python db/03_cargar_datos_iniciales.py  # carga el histórico desde el Excel
```

En local esto cargó **8.264 registros** de producción (rango 2023-01-01 → 2025-12-10), **1.278**
recetas y **859** de inventario. Con eso, `/api/historial_consumo` devuelve 2.594 registros para 2025.

⚠️ **Y recuerda la `SECRET_KEY`.** El archivo `.env` también está en `.gitignore` (correctamente: son
credenciales). Sin `SECRET_KEY` la aplicación **ni siquiera arranca**:

```python
if not app.secret_key:
    raise RuntimeError("SECRET_KEY no definida. Revisa tu archivo .env")
```

En el servidor hay que definirla como **variable de entorno**, no subirla al repositorio.

**Mejora adicional aplicada.** La tabla `intentos_login` ahora **se crea sola al arrancar la app**. Si
no, en un despliegue nuevo el bloqueo de 24 h fallaba **en silencio** (la consulta reventaba y el
contador nunca subía), que era justo la tercera hipótesis que esta guía mencionaba antes:

```python
# la tabla de intentos debe existir siempre, si no el bloqueo de 24h falla en
# silencio (p. ej. en un despliegue nuevo donde nadie corrio el script).
def asegurar_tabla_intentos():
    try:
        with conectar() as conexion:
            conexion.execute(
                """
                CREATE TABLE IF NOT EXISTS intentos_login (
                    clave TEXT PRIMARY KEY,
                    intentos INTEGER NOT NULL DEFAULT 0,
                    bloqueado_hasta REAL NOT NULL DEFAULT 0
                )
                """
            )
            conexion.commit()
    except Exception:
        logging.exception("No se pudo asegurar la tabla intentos_login")

asegurar_tabla_intentos()
```


---

### Error 5 — `pip install -r requirements.txt` falla entero

**Síntoma.** Al instalar las dependencias, **todo** el comando falla y ni siquiera Flask queda
instalado, así que la app no arranca.

**Causa raíz.** El archivo `requirements.txt` fija esta versión:

```
numpy>=1.24,<2.0
```

Ese `numpy` antiguo **no compila en Python 3.13** (la versión instalada en este equipo es la 3.13.7).
Y como `pip` instala todo o nada, **un solo paquete roto tumba la instalación completa** — incluidos
Flask, Werkzeug y flask-limiter, que no tienen ningún problema.

**Corrección.** Relajar o quitar ese pin. Si el proyecto no usa `numpy` directamente en la aplicación
web, se puede eliminar; si se necesita, basta con permitir versiones nuevas:

```
numpy>=1.24
```

Mientras tanto, el stack de la aplicación se puede instalar aparte:

```bash
pip install "Flask>=2.3" "flask-cors>=4.0" "Werkzeug>=2.3" "python-dotenv>=1.0" "flask-limiter>=3.5" "flask-wtf>=1.2"
```

> 💡 **Para la sección de recomendaciones:** este tipo de fallo es un buen ejemplo de por qué conviene
> fijar versiones **con criterio**. Un pin demasiado estricto (`<2.0`) protege de cambios que rompen,
> pero envejece mal: al cabo de un tiempo deja de compilar en los Python nuevos y **rompe el proyecto
> entero** para quien lo clone.


---

### 4.6 — Resumen de archivos tocados

| Archivo | Qué se cambió |
|---|---|
| `auth/login.py` | `_registrar_fallo()` avisa si acaba de bloquear; `mensaje_bloqueo()` habla en **horas**; nueva `estado_bloqueo_ip()` para que la página consulte el bloqueo al cargar |
| `app.py` | Límite por minuto subido a 20 (deja de tapar el bloqueo); errores 429 en **JSON**; `/login` consulta el bloqueo y lo renderiza; nuevo `/api/estado_bloqueo`; `/login` sin caché; `intentos_login` se crea al arrancar |
| `templates/login.html` | El botón y el mensaje se pintan según el **servidor**; el envío se corta si está bloqueado; revalidación al volver con el botón "atrás" (bfcache) |
| `requirements.txt` | Pin de `numpy` a revisar (rompe la instalación en Python 3.13) |
| *(despliegue)* | Ejecutar los scripts de `db/` en el servidor y definir `SECRET_KEY` como variable de entorno |

### 4.7 — Conclusión para la defensa

La medida de seguridad **existía y su motor estaba bien diseñado**: contador persistente en base de
datos, con clave `usuario@ip`, que sobrevive a reinicios del servidor y combina dos protecciones
complementarias (límite por *ritmo* + bloqueo por *cantidad*).

Lo que fallaba estaba **alrededor** de esa lógica, y es exactamente el tipo de error que aparece al
integrar piezas que por separado funcionan bien:

1. Dos capas de seguridad **compitiendo** entre sí (el límite por minuto tapaba al bloqueo).
2. El aviso llegando **un paso tarde**.
3. El estado de seguridad **confiado al navegador**, que el usuario puede reiniciar con un F5.

El tercero es el más instructivo: enseña que **la interfaz no es una medida de seguridad**. Deshabilitar
un botón es amabilidad con el usuario, no protección contra un atacante. La protección real es que el
servidor diga que no —y aquí lo dice con **HTTP 423, incluso cuando la contraseña es correcta**.


## 5. Objetivo específico 1 — Módulo de autenticación y autorización (RBAC)

> *Objetivo: “Diseñar un módulo de autenticación y autorización basado en RBAC… con gestión segura de sesiones y mitigación de accesos no autorizados.”*

**RBAC** = *Role-Based Access Control* (control de acceso **basado en roles**). La idea: cada usuario tiene un **rol**, y el rol define **qué puede hacer**. Se divide en dos preguntas:

- **Autenticación** = *¿eres quien dices ser?* (login con usuario y contraseña).
- **Autorización** = *¿tienes permiso para esto?* (¿tu rol te deja entrar a esta pantalla o acción?).

### 5.1 — Autenticación: las contraseñas NUNCA se guardan en texto plano
En `auth/login.py`:

```python
from werkzeug.security import generate_password_hash, check_password_hash

def hashear_password(password: str) -> str:
    ...
    return generate_password_hash(password)      # guarda un "hash", no la clave

def verificar_password(password, password_hash) -> bool:
    return check_password_hash(password_hash, password)   # compara sin des-hashear
```

**En sencillo:** un *hash* es una huella digital irreversible de la contraseña. En la base de datos se guarda **la huella**, no la clave. Al iniciar sesión, se calcula la huella de lo que escribes y se compara con la guardada. Si alguien robara la base de datos, **no vería las contraseñas**. Puedes verlo en vivo:

In [ ]:
from auth.login import hashear_password, verificar_password

clave = "Admin123!"
huella = hashear_password(clave)
print("Lo que se GUARDA en la base de datos (el hash):")
print(" ", huella[:70], "...\n")
print("¿Coincide con la clave correcta? ->", verificar_password("Admin123!", huella))
print("¿Coincide con una clave falsa?   ->", verificar_password("otra_clave", huella))

### 5.2 — Gestión **segura de sesiones**
Una vez inicias sesión, el servidor te da una *cookie* de sesión. El proyecto la protege así (en `app.py`):

```python
app.config["SESSION_COOKIE_HTTPONLY"] = True     # la cookie no es accesible desde JavaScript (anti-robo/XSS)
app.config["SESSION_COOKIE_SAMESITE"] = "Lax"    # no se envía en peticiones de otros sitios (ayuda anti-CSRF)
csrf = CSRFProtect(app)                            # protección CSRF en formularios
```

Y además:
- **Expiración por inactividad (15 min):** en `verificar_inactividad()` — si pasas 15 minutos sin actividad, la sesión se cierra sola.
- **Cabeceras de seguridad:** `X-Frame-Options: DENY` (anti *clickjacking*), `X-Content-Type-Options: nosniff`, `Cache-Control: no-store` en páginas sensibles, etc. (función `agregar_headers_seguridad`).
- **Límite de intentos por minuto** y **bloqueo de 24 h** (lo vimos en la sección 4).

En sencillo, esto significa: la sesión es difícil de robar (HttpOnly), no se puede usar desde otra web (SameSite + CSRF), no queda “abierta para siempre” (expira por inactividad) y no se puede “adivinar la clave a la fuerza” (límites + bloqueo).

### 5.3 — Autorización: los **decoradores** que protegen cada pantalla
En `auth/roles.py` se definen unos “sellos” (llamados *decoradores*) que se ponen encima de cada ruta para exigir un rol:

```python
def rol_requerido(*roles_permitidos):
    def decorador(f):
        @wraps(f)
        def envoltura(*args, **kwargs):
            if "usuario_id" not in session:            # ¿inició sesión?
                return _no_autenticado_response()      #   -> no: al login (o 401)
            if session.get("rol") not in roles_permitidos:  # ¿su rol tiene permiso?
                return _sin_permiso_response()         #   -> no: acceso denegado (o 403)
            return f(*args, **kwargs)                   #   -> sí: adelante
        return envoltura
    return decorador

def solo_admin(f):        return rol_requerido("Admin")(f)
def admin_u_operador(f):  return rol_requerido("Admin", "Operador")(f)
def cualquier_usuario(f): return rol_requerido("Admin","Operador","Visualizador")(f)
```

Y así se **aplican** en `app.py` (mira el “sello” encima de cada función):

```python
@app.route("/inventario")
@solo_admin                 # <- SOLO Admin puede entrar aquí
def inventario(): ...

@app.route("/historial")
@admin_u_operador           # <- Admin u Operador
def historial(): ...

@app.route("/dashboard")
@cualquier_usuario          # <- cualquiera con sesión válida
def dashboard(): ...
```

**En sencillo:** cada pantalla lleva un cartel de “quién puede pasar”. Si un *Visualizador* intenta abrir `/inventario`, el guardia lo devuelve al dashboard y **registra el intento** en el log de seguridad. Además, cada intento sin permiso queda anotado (para auditoría).

> 🧠 **Dato clave para la defensa:** la autorización se hace **en el servidor**, no en el navegador. Aunque alguien manipule la pantalla, el servidor vuelve a comprobar el rol en cada petición. Por eso es una mitigación **real** de accesos no autorizados.

## 6. Objetivo específico 2 — Consultas parametrizadas vs Inyección SQL

> *Objetivo: “Verificar la efectividad de las consultas parametrizadas y otras defensas contra Inyección SQL.”*

### 6.1 — ¿Qué es la inyección SQL? (en 30 segundos)
La base de datos entiende un idioma llamado **SQL**. Una consulta para buscar un usuario podría ser:

```sql
SELECT * FROM usuarios WHERE username = 'admin'
```

La **inyección SQL** ocurre cuando un programa **pega directamente** lo que escribió el usuario dentro de la consulta. Un atacante podría escribir como “usuario”:

```
' OR '1'='1
```

y si el código lo pega tal cual, la consulta se convierte en `... WHERE username = '' OR '1'='1'`, que es **siempre verdadera** → el atacante entra sin credenciales. Es una de las vulnerabilidades más graves y comunes.

### 6.2 — Cómo lo evita este proyecto: el signo `?` (consultas parametrizadas)
La defensa es **no pegar** el texto del usuario dentro del SQL. En su lugar se pone un **`?`** como hueco, y el valor viaja **por separado**. Así el motor trata el texto como **dato**, nunca como **instrucción**. Ejemplo real en `auth/login.py`:

```python
cursor.execute(
    "SELECT id, username, rol, password_hash FROM usuarios WHERE username = ?",
    (username,),           # el valor va aquí, aparte; el '?' es el hueco
)
```

Aunque el atacante escriba `' OR '1'='1`, el motor busca literalmente un usuario llamado *“' OR '1'='1”*, no lo encuentra, y **no pasa nada**. Puedes comprobarlo:

In [ ]:
from auth.login import autenticar

# Intento de inyección SQL clásico en el campo usuario:
ataque = "' OR '1'='1"
r = autenticar(ataque, "lo_que_sea", ip="127.0.0.1")
print("Usuario malicioso probado:", repr(ataque))
print("Resultado ->", r)
print("\n➡️  No entra: el '?' hizo que el texto se tratara como dato, no como comando SQL.")

Este patrón se repite en **toda** la capa de datos. Por ejemplo, en `services/historial.py` los filtros se arman con `?` y una lista de parámetros aparte:

```python
condiciones = ["pd.fecha >= ? AND pd.fecha <= ?"]
parametros  = [inicio, fin]
if diseno:
    condiciones.append("m.diseno_mezcla = ?")   # hueco
    parametros.append(diseno)                    # valor aparte
if zona:
    condiciones.append("z.nombre_zona LIKE ?")
    parametros.append(f"%{zona}%")
...
sql = f"... WHERE {' AND '.join(condiciones)} ..."
cursor.execute(sql, parametros)     # los valores SIEMPRE viajan aparte
```

Aquí lo único que se “arma” dinámicamente son **nombres de columnas fijos escritos por el programador** (no texto del usuario); **los valores** del usuario siempre van como parámetros `?`.

### 6.3 — La “lista blanca” para lo que no se puede parametrizar
Hay un detalle técnico: el `?` sirve para **valores**, pero **no** para nombres de **tablas o columnas**. En los pocos sitios donde hay que poner un nombre de tabla dentro del SQL, el proyecto usa una **lista blanca** (una lista cerrada de nombres permitidos). En `utils/db.py`:

```python
_TABLAS_PERMITIDAS = frozenset({
    "despachos", "materiales", "usuarios", "Insumos", "Zonas",
    "Produccion_Diaria", "intentos_login", ...
})

def columnas_tabla(conexion, tabla):
    if tabla not in _TABLAS_PERMITIDAS:          # si no está en la lista...
        raise ValueError(f"Tabla '{tabla}' no permitida.")   # ...se rechaza
    cursor = conexion.execute(f"PRAGMA table_info({tabla})")
    ...
```

**En sencillo:** como aquí no se puede usar `?`, se valida el nombre contra una **lista de nombres autorizados**. Si el nombre no está en la lista, se rechaza. Así ni siquiera por ahí se puede “colar” algo peligroso.

> ✅ **Conclusión para la defensa:** el uso **sistemático** de consultas parametrizadas impide que la entrada del usuario altere la lógica del SQL; y donde no se puede parametrizar (nombres de tabla), una **lista blanca** cierra el hueco. Por eso los puntos de entrada evaluados **no resultaron vulnerables**.

## 7. Objetivo específico 3 — Normalización hasta 3FN

> *Objetivo: “Demostrar el proceso de normalización desde un modelo desnormalizado hasta 3FN.”*

**Normalizar** una base de datos = **organizar los datos** para que **no se repitan** y **no haya inconsistencias**. Se hace por etapas llamadas *formas normales* (1FN → 2FN → 3FN).

### 7.1 — El punto de partida: modelo DESNORMALIZADO
El diseño inicial (ver `docs/diagrama_relacional_original.mermaid`) tenía prácticamente **3 tablas gigantes**:

- `PRODUCCION` con todo mezclado: fecha, turno, zona, wbs, diseño, volumen… (los catálogos escritos “a mano” en cada fila).
- `RECETA` con **una columna por cada insumo**: `arena_kg`, `grava_kg`, `cemento_kg`, `agua_l`, `glenium_ml`, `silka_ml`, `rheo1000_ml`, … (¡un “grupo repetitivo” de columnas!).
- `MATERIALES` con el stock.

**Problemas de ese diseño** (esto es justo lo que dice tu documento):
- **Redundancia:** el nombre del turno, la zona o el WBS se repetían en cada despacho.
- **Anomalías de actualización:** para corregir el nombre de una zona había que tocar **todas** las filas de producción.
- **Dependencias parciales:** los insumos de la receta dependen **solo del diseño**, no del despacho, pero vivían en la misma tabla.
- **Sin trazabilidad:** no había usuario, auditoría ni control de accesos.

### 7.2 — Las tres formas normales, con el ejemplo del proyecto

**1FN (Primera Forma Normal): eliminar los “grupos repetitivos”.**
La tabla `RECETA` tenía muchas columnas de insumos (`arena_kg`, `cemento_kg`, …). En 1FN eso se convierte en **filas**, no columnas: aparece la tabla **`Receta_Detalle`** donde cada fila es *(diseño, insumo, cantidad)*. Así puedes añadir un insumo nuevo **sin cambiar la estructura** de la tabla.

**2FN (Segunda Forma Normal): eliminar dependencias parciales.**
“Dependencia parcial” = un dato que depende **solo de una parte** de la clave. Los ingredientes dependían **solo del diseño de mezcla**, no del despacho concreto. Al separar la **receta** (qué lleva un diseño) del **consumo real** de cada despacho (`Produccion_Insumos`), cada dato queda con la clave de la que realmente depende.

**3FN (Tercera Forma Normal): eliminar dependencias transitivas.**
“Dependencia transitiva” = un dato que depende de otro dato que **no es la clave**. Los nombres de **zona, turno y centro de costo (WBS)** se sacan a **tablas de catálogo** propias (`Zonas`, `Turnos`, `Centros_Costo`), y en producción se guarda solo su **identificador (id)**. Cambiar el nombre de una zona ahora es **un solo cambio** en el catálogo.

### 7.3 — El resultado: 11 tablas en 3FN
El esquema real (ver `docs/diagrama_relacional_3FN.mermaid`, y el código en `db/01_crear_esquema.py`) tiene **11 tablas**:

| # | Tabla | Para qué | Rol en la normalización |
|---|---|---|---|
| 1 | `usuarios` | Usuarios del sistema | Seguridad / RBAC |
| 2 | `intentos_login` | Control de intentos y bloqueo | Seguridad |
| 3 | `Insumos` | Catálogo de materiales + stock | Entidad principal |
| 4 | `movimientos` | Auditoría de cambios de inventario | Trazabilidad |
| 5 | `Zonas` | Catálogo de zonas | Catálogo (quita redundancia) |
| 6 | `Centros_Costo` | Catálogo de WBS | Catálogo |
| 7 | `Turnos` | Catálogo de turnos | Catálogo |
| 8 | `Disenos_Mezcla` | Cabecera de recetas | Entidad |
| 9 | `Receta_Detalle` | Ingredientes por diseño (N:M) | Resuelve grupo repetitivo (1FN) |
| 10 | `Produccion_Diaria` | Cabecera de despachos | Entidad, con **FK** a los catálogos |
| 11 | `Produccion_Insumos` | Consumo real por despacho (N:M) | **Clave primaria compuesta** |

**Claves (para tu conclusión):**
- **Claves primarias simples:** p. ej. `Zonas.id_zona`.
- **Clave primaria compuesta:** `Produccion_Insumos (id_produccion, id_insumo)` — la combinación de dos ids identifica cada fila.
- **Claves foráneas (FK):** p. ej. `Produccion_Diaria.id_zona` apunta a `Zonas.id_zona`, garantizando **integridad referencial** (no puedes registrar un despacho en una zona que no existe).

La siguiente celda **te muestra las 11 tablas reales** de tu base (necesita haber hecho el Paso 3.3):

In [ ]:
import sqlite3
con = sqlite3.connect("db/gestion_materiales.db")
tablas = [r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name")]
print("Tablas en la base de datos (", len(tablas), "):\n")
for t in tablas:
    cols = [c[1] for c in con.execute(f"PRAGMA table_info({t})")]
    print(f"• {t}: {', '.join(cols)}")
con.close()

> 📎 **Diagramas:** en la carpeta `docs/` tienes los tres diagramas listos para tu documento:
> `diagrama_relacional_original.mermaid` (antes), `diagrama_relacional_2FN.mermaid` (paso intermedio) y `diagrama_relacional_3FN.mermaid` (resultado). Se pueden ver en cualquier visor de *Mermaid* (o en VS Code con la extensión de Mermaid).

> ✅ **Conclusión para la defensa:** el modelo inicial, con grupos repetitivos y dependencias parciales, se transformó hasta **3FN** separando entidades en **catálogos**, usando **claves primarias** (simples y compuestas) y **foráneas**, y eliminando dependencias parciales y transitivas. El resultado es un esquema **consistente, sin redundancias y con integridad referencial garantizada**.

## 8. Mapa: tus conclusiones y recomendaciones ↔ el código

Para tu defensa, aquí tienes **dónde “se toca” en el código** cada afirmación de tu documento.

**Conclusiones**

| Tu conclusión | Dónde está en el código |
|---|---|
| Consultas parametrizadas impiden alterar la lógica SQL | `auth/login.py` (`buscar_usuario`), `services/historial.py`, `services/inventario.py` — uso de `?` |
| Lista blanca donde no se puede parametrizar | `utils/db.py` → `_TABLAS_PERMITIDAS` + `columnas_tabla()` |
| Sesiones seguras (HttpOnly, SameSite, CSRF, expiración, límite/min, bloqueo persistente) | `app.py` (config de cookies, `CSRFProtect`, `verificar_inactividad`, `@limiter.limit`) + `auth/login.py` (bloqueo 24 h) |
| Restricción de operaciones según rol (RBAC) | `auth/roles.py` (decoradores) aplicados en `app.py` |
| Credenciales solo como hash, nunca en texto plano | `auth/login.py` (`hashear_password` / `verificar_password`) |
| Normalización hasta 3FN, PK/FK, sin redundancia | `db/01_crear_esquema.py` + `docs/diagrama_relacional_3FN.mermaid` |

**Recomendaciones** (todas son coherentes con lo que hay hoy)

| Tu recomendación | Comentario técnico |
|---|---|
| Añadir **MFA** (doble factor), sobre todo a Admin | Se sumaría al login actual; hoy no está implementado |
| **Sesión única por usuario** (invalidar la anterior al entrar en otro dispositivo) | Hoy no hay control de sesión única; requeriría guardar un identificador de sesión activo por usuario |
| Migrar de **SQLite** a **PostgreSQL/SQL Server** (o MongoDB si aplica) | Hoy usa SQLite (`utils/db.py`, `RUTA_BD`); mejoraría concurrencia y administración |
| **Mensajes de error genéricos** al usuario | Ya se aplica: `MSG_ERROR_GENERICO` en `app.py`; se puede extender a la ruta de login |

## 9. Glosario para perfil no técnico

- **Backend / servidor:** el “cerebro” que corre en Python y decide qué se puede hacer. El usuario no lo ve.
- **Flask:** el framework (kit de herramientas) con el que está hecho el servidor.
- **SQLite:** la base de datos guardada en un solo archivo (`.db`).
- **Endpoint / ruta:** una dirección de la app (p. ej. `/login`, `/inventario`) que hace algo.
- **RBAC:** control de acceso por **roles** (Admin, Operador, Visualizador).
- **Autenticación:** verificar **quién eres** (login).
- **Autorización:** verificar **qué puedes hacer** (permisos por rol).
- **Hash:** “huella digital” irreversible de la contraseña; se guarda la huella, no la clave.
- **Sesión / cookie:** el “pase” que te identifica mientras navegas ya logueado.
- **CSRF:** ataque que hace que tu navegador envíe acciones sin que quieras; se mitiga con token + SameSite.
- **HttpOnly:** propiedad de la cookie para que **JavaScript no pueda leerla** (dificulta el robo).
- **Inyección SQL:** meter comandos maliciosos por un campo de texto; se evita con **consultas parametrizadas** (`?`).
- **Consulta parametrizada:** el valor del usuario viaja **aparte** del comando SQL; el motor lo trata como dato.
- **Lista blanca:** lista cerrada de valores permitidos; todo lo que no está, se rechaza.
- **Fuerza bruta:** probar miles de contraseñas hasta acertar; se mitiga con **límite de intentos + bloqueo**.
- **Normalización / 3FN:** organizar los datos en tablas para evitar repetición e inconsistencias.
- **Clave primaria (PK):** identificador único de cada fila.
- **Clave foránea (FK):** columna que apunta a la PK de otra tabla (garantiza integridad).
- **Catálogo:** tabla pequeña con valores de referencia (zonas, turnos, etc.).

---
*Cuaderno generado como apoyo para la defensa. Todas las rutas de archivo (`auth/login.py`, `app.py`, `db/01_crear_esquema.py`, etc.) corresponden a esta copia del proyecto. Las celdas de código son reproducibles tras ejecutar el Paso 3.*